# Module 11 — What the loop costs

**THE ONE IDEA:** an agent costs roughly **10x a single call**, and the reason is
arithmetic you can watch: **every turn re-sends the entire conversation.**

Module 02 showed input tokens climbing across a chat. Here the same effect runs inside a
tool loop, where each turn also carries the tool schemas and every prior tool result.

This closes Block C. Blocks D through F exist because this number, and the failures that
inflate it, are what stop an agent shipping.


In [1]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json
from _providers import get_client
from _tools import openai_schemas, run_tool

client, MODEL, _ = get_client("openai")
PRICE_IN, PRICE_OUT = 0.40, 1.60          # $ per 1M tokens, gpt-4.1-mini

QUESTION = ("What is the early repayment charge in year 2 on a 250000 loan, "
            "and how does it compare to the year 5 charge?")

def cost(tin, tout):
    return tin * PRICE_IN / 1e6 + tout * PRICE_OUT / 1e6

## Baseline — one call, no tools

In [2]:
r = client.chat.completions.create(model=MODEL, max_tokens=400,
                                   messages=[{"role": "user", "content": QUESTION}])
b_in, b_out = r.usage.prompt_tokens, r.usage.completion_tokens
base = cost(b_in, b_out)
print(f"single call: in={b_in} out={b_out} cost=${base:.6f}")
print("(it answers from memory — possibly wrong, but it is the cost floor)")

single call: in=36 out=400 cost=$0.000654
(it answers from memory — possibly wrong, but it is the cost floor)


## The agent loop, metered per turn

In [3]:
messages = [{"role": "user", "content": QUESTION}]
rows = []
for step in range(1, 8):
    r = client.chat.completions.create(model=MODEL, max_tokens=500,
                                       tools=openai_schemas(["search_policy", "calculate"]),
                                       messages=messages)
    msg = r.choices[0].message
    tin, tout = r.usage.prompt_tokens, r.usage.completion_tokens
    rows.append((step, len(messages), tin, tout, cost(tin, tout)))
    if r.choices[0].finish_reason != "tool_calls":
        answer = msg.content
        break
    messages.append(msg)
    for tc in msg.tool_calls:
        messages.append({"role": "tool", "tool_call_id": tc.id,
                         "content": run_tool(tc.function.name,
                                             json.loads(tc.function.arguments))})

print(f"{'turn':>4} {'msgs':>5} {'in':>7} {'out':>6} {'$ this turn':>13}")
print("-" * 40)
for s, m, tin, tout, c in rows:
    print(f"{s:4} {m:5} {tin:7} {tout:6} {c:13.6f}")

turn  msgs      in    out   $ this turn
----------------------------------------
   1     1     124     68      0.000158
   2     5     260     14      0.000126
   3     7     316     54      0.000213
   4    10     392     73      0.000274


## The bill

In [4]:
agent_in  = sum(r[2] for r in rows)
agent_out = sum(r[3] for r in rows)
agent     = sum(r[4] for r in rows)

print(f"{'':16} {'in':>8} {'out':>7} {'$':>11}")
print("-" * 46)
print(f"{'single call':16} {b_in:8} {b_out:7} {base:11.6f}")
print(f"{'agent loop':16} {agent_in:8} {agent_out:7} {agent:11.6f}")
print("-" * 46)
print(f"{'multiplier':16} {agent_in / b_in:7.1f}x {agent_out / b_out:6.1f}x "
      f"{agent / base:10.1f}x")
print(f"\nturns: {len(rows)}   first-turn input {rows[0][2]} -> last-turn input {rows[-1][2]}")

print("""
LESSON — read the INPUT multiplier, not the total. Input is where the agent
tax lives, and three causes compound:
  1. the tool SCHEMAS ride along on every single turn
  2. every prior assistant turn and tool result is re-sent (module 02)
  3. more turns means more of 1 and 2

TOTAL cost is a different question, because output is priced ~4x input and the
baseline is free to be verbose. A single call that rambles for 300 tokens can
out-cost a 2-turn agent that answers in 100 — and when that happens the total
multiplier drops BELOW 1x even though the input multiplier is 10x. If your run
shows that, it is not a broken demo: it is the honest shape of the trade.

The famous '~10x a single call' figure assumes a LONG loop. Scale the turn
count and the input curve dominates everything else, because it grows roughly
with the SQUARE of turns. That is why module 13's guards are budget guards as
much as safety guards, and why module 22 starts throwing the list away.

Two levers before anything clever: prompt CACHING on the stable prefix
(schemas + system), and ROUTING easy work to a cheap model (module 05).

Run this on claude-opus-5 and it gets worse — thinking is billed as output on
EVERY turn. Module 01 measured 632 output tokens for a one-sentence answer.""")

                       in     out           $
----------------------------------------------
single call            36     400    0.000654
agent loop           1092     209    0.000771
----------------------------------------------
multiplier          30.3x    0.5x        1.2x

turns: 4   first-turn input 124 -> last-turn input 392

LESSON — read the INPUT multiplier, not the total. Input is where the agent
tax lives, and three causes compound:
  1. the tool SCHEMAS ride along on every single turn
  2. every prior assistant turn and tool result is re-sent (module 02)
  3. more turns means more of 1 and 2

TOTAL cost is a different question, because output is priced ~4x input and the
baseline is free to be verbose. A single call that rambles for 300 tokens can
out-cost a 2-turn agent that answers in 100 — and when that happens the total
multiplier drops BELOW 1x even though the input multiplier is 10x. If your run
shows that, it is not a broken demo: it is the honest shape of the trade.

---

**Next:** `../D_reliability/12_failure_modes_reproduced.ipynb`